# AntibioticCheck — Multi-Agent Scientific Evidence Review

Adapted from the uploaded **ABX-AI v9** notebook. The original retrieval, passage ranking, citation/context audit, conservative decision logic, and evaluation helpers are preserved where possible. Local transformer NLI and sentence-transformer embeddings are replaced with hosted Google Gemma 4 (`gemma-4-26b-a4b-it`) via Google GenAI.

**Research prototype only — not medical advice.**

## 1. Setup and API key

Install only the lightweight packages below. No model weights are downloaded. Set `GOOGLE_API_KEY` in your environment or enter it when prompted. Never save your API key in this notebook or commit it to GitHub.

In [ ]:
!pip -q install requests pandas numpy rank-bm25 scikit-learn reportlab google-genai
import os, getpass
if not (os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Google AI Studio API key (input hidden): ").strip()


## 2. Configuration and benchmark

Benchmark execution is disabled by default to avoid unexpected API usage. Turn it on only when you want to run the full set.

In [ ]:
import os, re, json, time, html, hashlib, platform
from difflib import SequenceMatcher
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError
import numpy as np
import pandas as pd
import requests
from rank_bm25 import BM25Okapi
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet
from xml.sax.saxutils import escape

SEED = 42
CLAIM_ID = "ABX-001"
CLAIM_TO_CHECK = "Antibiotics do not treat viral infections."
MAX_RESULTS_PER_QUERY = 25
MAX_PAPERS_TO_RANK = 20
MAX_EVIDENCE_PASSAGES = 6
REQUEST_TIMEOUT_SECONDS = 30
RUN_BENCHMARK = False
BENCHMARK_LIMIT = None  # Set to 3 for a quicker smoke test.
# True means each benchmark claim attempts Gemini when a key is available.
# Set False to save API calls; the output will clearly state that LLM was disabled.
BENCHMARK_USE_LLM = True
NLI_MODEL_NAME = "gemma-4-26b-a4b-it"
EMBEDDING_MODEL_NAME = "BM25-only (no local embeddings)"
# Override with GEMINI_MODEL_NAME environment variable if your API account uses another supported model.
GEMINI_MODEL_NAME = os.environ.get("GEMINI_MODEL_NAME", "gemma-4-26b-a4b-it")
# Fail over only on transient service/capacity failures; first model remains the preferred choice.
GEMINI_FALLBACK_MODELS = [
    m.strip() for m in os.environ.get(
        "GEMINI_FALLBACK_MODELS",
        ""
    ).split(",") if m.strip() and m.strip() != GEMINI_MODEL_NAME
]
GEMINI_MAX_RETRIES_PER_MODEL = 2
GEMINI_BACKOFF_BASE_SECONDS = 1.0
GEMINI_BACKOFF_MAX_SECONDS = 4.0

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "").strip() or os.environ.get("GOOGLE_API_KEY", "").strip()
try:
    from kaggle_secrets import UserSecretsClient
    if not GEMINI_API_KEY:
        try:
            GEMINI_API_KEY = UserSecretsClient().get_secret("GEMINI_API_KEY").strip()
        except Exception as secret_exc:
            print("Kaggle secret unavailable; checking environment variables. Detail:", str(secret_exc)[:180])
except Exception:
    pass
LLM_ENABLED = bool(GEMINI_API_KEY)
DEVICE = "API-only (no local model)"
OUT_DIR = Path(os.environ.get("ABX_OUTPUT_DIR", "outputs"))
OUT_DIR.mkdir(parents=True, exist_ok=True)
np.random.seed(SEED)

def package_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return "not-installed"

RUN_STARTED_UTC = datetime.now(timezone.utc).isoformat()
ENVIRONMENT = {
    "python": platform.python_version(), "pandas": package_version("pandas"),
    "numpy": package_version("numpy"), "requests": package_version("requests"),
    "google-genai": package_version("google-genai"),
    "rank_bm25": package_version("rank-bm25"),
    
    "inference_mode": DEVICE, "nli_model": NLI_MODEL_NAME,
    "embedding_model": EMBEDDING_MODEL_NAME,
    "llm_enabled": LLM_ENABLED,
    "llm_model": GEMINI_MODEL_NAME if LLM_ENABLED else "disabled_no_api_key",
    "llm_fallback_models": GEMINI_FALLBACK_MODELS,
    "benchmark_use_llm": BENCHMARK_USE_LLM,
    "seed": SEED, "run_started_utc": RUN_STARTED_UTC
}
print(json.dumps(ENVIRONMENT, indent=2))

# Google GenAI SDK client: inference is hosted; no local model weights are downloaded.
try:
    from google import genai
except Exception:
    genai = None

def get_genai_client():
    key = (os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY") or GEMINI_API_KEY or "").strip()
    if not key:
        raise RuntimeError("Missing Google API key. Set GOOGLE_API_KEY or GEMINI_API_KEY.")
    if genai is None:
        raise RuntimeError("google-genai is not installed. Run pip install -r requirements.txt.")
    return genai.Client(api_key=key)


## 3. Curated starter benchmark

In [ ]:
BENCHMARK_CLAIMS = [
 {"claim_id":"ABX-001","claim":"Antibiotics do not treat viral infections.","reference_label":"SUPPORTED","reference_source_url":"https://www.cdc.gov/antibiotic-use/about/","reference_note":"CDC states antibiotics do not work on viruses.","topic":"antibiotics_viral"},
 {"claim_id":"ABX-002","claim":"Antibiotics treat the common cold.","reference_label":"CONTRADICTED","reference_source_url":"https://www.cdc.gov/common-cold/treatment/index.html","reference_note":"CDC states antibiotics do not work against viruses causing the common cold.","topic":"antibiotics_viral"},
 {"claim_id":"ABX-003","claim":"Antibiotics treat influenza viruses.","reference_label":"CONTRADICTED","reference_source_url":"https://www.cdc.gov/flu/treatment/index.html","reference_note":"CDC distinguishes antiviral medicines from antibiotics.","topic":"antibiotics_viral"},
 {"claim_id":"ABX-004","claim":"Antibiotics can treat some infections caused by bacteria.","reference_label":"SUPPORTED","reference_source_url":"https://www.cdc.gov/antibiotic-use/about/","reference_note":"CDC lists selected bacterial infections treated with antibiotics.","topic":"antibiotics_bacterial"},
 {"claim_id":"ABX-005","claim":"Every infection caused by bacteria always requires antibiotics.","reference_label":"CONTRADICTED","reference_source_url":"https://www.cdc.gov/antibiotic-use/about/","reference_note":"Some bacterial infections may improve without antibiotics; they are not always needed.","topic":"antibiotics_bacterial"},
 {"claim_id":"ABX-006","claim":"Misuse and overuse of antimicrobials drive antimicrobial resistance.","reference_label":"SUPPORTED","reference_source_url":"https://www.who.int/news-room/fact-sheets/detail/antimicrobial-resistance","reference_note":"WHO identifies misuse and overuse as drivers of resistance.","topic":"amr"},
 {"claim_id":"ABX-007","claim":"Taking antibiotics when they are not needed can contribute to antibiotic resistance.","reference_label":"SUPPORTED","reference_source_url":"https://www.cdc.gov/antibiotic-use/data-research/facts-stats/","reference_note":"CDC explains that unnecessary antibiotic use contributes to resistance.","topic":"amr"},
 {"claim_id":"ABX-008","claim":"Antibiotic resistance means a person's body becomes resistant to antibiotics.","reference_label":"CONTRADICTED","reference_source_url":"https://www.cdc.gov/antibiotic-use/media/pdfs/Antibiotics-Arent-Always-the-Answer-508.pdf","reference_note":"CDC clarifies that bacteria, not the person's body, develop resistance.","topic":"amr"},
 {"claim_id":"ABX-009","claim":"Antibiotics can cause side effects even when they are not needed.","reference_label":"SUPPORTED","reference_source_url":"https://www.cdc.gov/antibiotic-use/about/","reference_note":"CDC notes unnecessary antibiotics can still cause harm.","topic":"side_effects"},
 {"claim_id":"ABX-010","claim":"Antivirals and antibiotics are the same type of medicine.","reference_label":"CONTRADICTED","reference_source_url":"https://www.cdc.gov/flu/treatment/index.html","reference_note":"CDC distinguishes antiviral drugs from antibiotics.","topic":"antivirals"},
 {"claim_id":"ABX-011","claim":"Yellow or green mucus by itself means antibiotics are needed.","reference_label":"CONTRADICTED","reference_source_url":"https://www.cdc.gov/antibiotic-use/about/","reference_note":"CDC says antibiotics do not treat colds/runny noses even when mucus is yellow or green.","topic":"symptoms"},
 {"claim_id":"ABX-012","claim":"A viral infection can sometimes be followed by a secondary bacterial infection that may require antibiotics.","reference_label":"CONTEXT_DEPENDENT","reference_source_url":"https://www.cdc.gov/common-cold/treatment/index.html","reference_note":"CDC notes a viral illness can sometimes lead to a bacterial infection; treatment depends on clinical assessment.","topic":"secondary_infection"}
]
benchmark_df = pd.DataFrame(BENCHMARK_CLAIMS)
benchmark_df["reference_status"] = "curated_starter_not_expert_adjudicated"
benchmark_df["gold_relevant_source_ids"] = ""
benchmark_path = OUT_DIR / "benchmark_manifest.csv"
benchmark_df.to_csv(benchmark_path, index=False)
display(benchmark_df)
print("Saved:", benchmark_path)

## 4. Shared text utilities and source identifiers

In [ ]:
EUROPE_PMC_URL = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"
SESSION = requests.Session()
SESSION.headers.update({"User-Agent":"ABX-AI-research-prototype/4.0 educational use"})
STOPWORDS = set("a an the is are was were be been being do does did not no and or to of for in on with as by it this that these those can could may might should would will how what why from into about after before between than then their them they there which while".split())

def clean_text(value):
    text = html.unescape(str(value or ""))
    text = re.sub(r"<[^>]+>", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def tokenize(text):
    return re.findall(r"[a-z0-9]+", clean_text(text).lower())

def source_identifier(record):
    for key, prefix in [("pmid","PMID:"),("pmcid","PMCID:"),("doi","DOI:")]:
        value = str(record.get(key) or "").strip()
        if value:
            return prefix + value.lower()
    title = re.sub(r"[^a-z0-9]+"," ",clean_text(record.get("title","")).lower()).strip()
    return "TITLE:" + hashlib.sha1(title.encode()).hexdigest()[:12] if title else ""

def source_url(record):
    if record.get("pmid"): return "https://pubmed.ncbi.nlm.nih.gov/" + str(record["pmid"]) + "/"
    if record.get("pmcid"): return "https://europepmc.org/articles/" + str(record["pmcid"])
    if record.get("doi"): return "https://doi.org/" + str(record["doi"])
    return ""

def year_from_record(record):
    raw = str(record.get("firstPublicationDate") or record.get("pubYear") or "")
    match = re.search(r"\b(?:19|20)\d{2}\b", raw)
    return match.group(0) if match else ""

## Agent 1 — Scope & Safety

Module: `src/agents/agent_01_scope_safety.py`

In [ ]:
PERSONAL_PATTERNS = [
 r"\bwhat dose\b", r"\bwhich antibiotic should i\b", r"\bshould i take\b",
 r"\bcan i take\b", r"\bwhat medicine should\b", r"\bmy symptoms\b",
 r"\bdiagnose me\b", r"\bfor my infection\b"
]
def scope_safety_agent(claim):
    text = clean_text(claim)
    if len(text) < 8 or len(text) > 1000:
        return {"allowed":False,"status":"invalid_input","reason":"Please provide a complete public claim under 1000 characters."}
    if any(re.search(p,text.lower()) for p in PERSONAL_PATTERNS):
        return {"allowed":False,"status":"personal_medical_request_blocked","reason":"This system checks public claims and cannot provide personal diagnosis or treatment advice."}
    return {"allowed":True,"status":"allowed_public_claim","reason":"Passed basic public-claim scope rules."}


## Agent 2 — Query Planning

Module: `src/agents/agent_02_query_planning.py`

In [ ]:
def query_planning_agent(claim):
    """Generate complementary high-recall queries; each query has a distinct retrieval role."""
    claim_clean = clean_text(claim)
    tokens = [t for t in tokenize(claim_clean) if t not in STOPWORDS and len(t) > 2]
    # Keep meaningful claim terms, but avoid building a brittle query from every token.
    focus = list(dict.fromkeys(tokens))[:8]
    antibiotic_terms = '(antibiotic OR antibiotics OR antibacterial OR "antimicrobial agent")'
    queries = [
        # Core topic query: maximize recall in title/abstract.
        f'TITLE_ABS:{antibiotic_terms} AND TITLE_ABS:({" OR ".join(focus)})' if focus
        else f'TITLE_ABS:{antibiotic_terms}',
        # Claim-focused clinical terms.
        f'({antibiotic_terms}) AND TITLE_ABS:({" OR ".join(focus)})' if focus
        else f'{antibiotic_terms}',
        # Wider abstract/full-text indexed search catches papers with less specific titles.
        f'({antibiotic_terms}) AND ({" OR ".join(focus)})' if focus
        else antibiotic_terms,
    ]
    # Add claim phrase search where the text is short enough to be useful.
    if len(claim_clean.split()) <= 12:
        phrase = claim_clean.replace('"', '').strip()
        queries.append(f'TITLE_ABS:"{phrase}"')
    # For the common antibiotic/viral-infection topic, include a high-recall clinical variant.
    tset = set(tokens)
    if tset & {"virus", "viruses", "viral", "influenza", "flu", "cold"}:
        queries.extend([
            'TITLE_ABS:(antibiotic OR antibiotics) AND TITLE_ABS:(viral OR viruses OR influenza OR "common cold")',
            'TITLE_ABS:(antibiotic OR antibiotics) AND TITLE_ABS:(antiviral OR bacterial OR coinfection OR "secondary infection")'
        ])
    # Avoid malformed or duplicate strings; cap API calls to keep runtime bounded.
    return list(dict.fromkeys(q for q in queries if q))[:5]


## Agent 3 — Europe PMC Retrieval

Module: `src/agents/agent_03_europe_pmc_retrieval.py`

In [ ]:
def europe_pmc_search(query, page_size=MAX_RESULTS_PER_QUERY):
    response = SESSION.get(EUROPE_PMC_URL, params={
        "query": query, "format": "json", "pageSize": int(page_size), "resultType": "core"
    }, timeout=REQUEST_TIMEOUT_SECONDS)
    response.raise_for_status()
    return response.json().get("resultList", {}).get("result", []) or []

def normalize_record(item):
    pubtypes = item.get("pubTypeList", {}).get("pubType", []) or []
    if isinstance(pubtypes, str):
        pubtypes = [pubtypes]
    rec = {
        "title": clean_text(item.get("title", "")),
        "abstract": clean_text(item.get("abstractText", "")),
        "year": year_from_record(item),
        "publication_date": str(item.get("firstPublicationDate") or ""),
        "journal": clean_text(item.get("journalTitle", "")),
        "authors": clean_text(item.get("authorString", "")),
        "publication_types": "; ".join(clean_text(x) for x in pubtypes if x),
        "is_open_access": str(item.get("isOpenAccess") or ""),
        "pmid": str(item.get("pmid") or "").strip(),
        "pmcid": str(item.get("pmcid") or "").strip(),
        "doi": str(item.get("doi") or "").strip()
    }
    rec["source_id"] = source_identifier(rec)
    rec["url"] = source_url(rec)
    return rec

def research_retriever_agent(queries):
    start = time.time()
    records, query_logs = [], []
    for query in queries:
        q_start = time.time()
        try:
            raw = europe_pmc_search(query)
            records.extend([normalize_record(x) for x in raw if x.get("title") or x.get("abstractText")])
            query_logs.append({"query": query, "status": "success", "raw_count": len(raw),
                               "elapsed_seconds": round(time.time()-q_start, 3), "error": ""})
        except Exception as exc:
            query_logs.append({"query": query, "status": "error", "raw_count": 0,
                               "elapsed_seconds": round(time.time()-q_start, 3), "error": str(exc)[:400]})
    unique, seen_ids, seen_titles = [], set(), set()
    for rec in records:
        sid = rec.get("source_id", "")
        title = re.sub(r"[^a-z0-9]+", " ", rec.get("title", "").lower()).strip()
        if sid and sid in seen_ids:
            continue
        if title and title in seen_titles:
            continue
        duplicate = False
        for old in unique:
            old_title = re.sub(r"[^a-z0-9]+", " ", old.get("title", "").lower()).strip()
            if title and old_title and SequenceMatcher(None, title, old_title).ratio() >= 0.93:
                duplicate = True
                break
        if duplicate:
            continue
        unique.append(rec)
        if sid: seen_ids.add(sid)
        if title: seen_titles.add(title)
    statuses = [x["status"] for x in query_logs]
    status = "success" if unique and "success" in statuses else ("empty" if "success" in statuses else "error")
    return {"status": status, "papers": unique, "retrieved_count": len(unique), "query_logs": query_logs,
            "elapsed_seconds": round(time.time()-start, 3),
            "error": "; ".join(x["error"] for x in query_logs if x["error"])}


## Agent 4 — Hybrid Ranking (BM25; no local embeddings)

Module: `src/agents/agent_04_hybrid_ranking.py`

In [ ]:
def _minmax(values):
    values = np.asarray(values, dtype=float)
    if len(values) == 0: return values
    lo, hi = float(np.min(values)), float(np.max(values))
    return (values-lo)/(hi-lo+1e-9)

def paper_quality_prior(paper):
    """Small, transparent prior; never treated as proof of truth."""
    text = (str(paper.get("title","")) + " " + str(paper.get("publication_types",""))).lower()
    prior = 0.0
    if "systematic review" in text or "meta-analysis" in text: prior += 0.06
    if "randomized controlled trial" in text or "randomised controlled trial" in text: prior += 0.04
    if "case report" in text or "editorial" in text or "letter" in text: prior -= 0.04
    try:
        year = int(str(paper.get("year",""))[:4])
        if year >= 2020: prior += 0.02
    except Exception:
        pass
    return float(np.clip(prior, -0.06, 0.08))

def hybrid_rank_papers(claim, papers, top_n=MAX_PAPERS_TO_RANK):
    """BM25 + transparent metadata prior; intentionally no local embedding model."""
    if not papers: return pd.DataFrame()
    docs = [(p.get("title","")+" "+p.get("abstract","")).strip() for p in papers]
    tokenized = [tokenize(x) for x in docs]
    query_tokens = [t for t in tokenize(claim) if t not in STOPWORDS]
    if not query_tokens or not any(tokenized): return pd.DataFrame()
    bm_raw = np.asarray(BM25Okapi(tokenized).get_scores(query_tokens), dtype=float)
    bm = _minmax(bm_raw); rows=[]
    for i, paper in enumerate(papers):
        prior = paper_quality_prior(paper)
        combined = float(np.clip(bm[i] + prior, 0, 1))
        rows.append({**paper, "bm25_score":float(bm_raw[i]), "bm25_score_norm":float(bm[i]),
                     "semantic_similarity_norm":None, "quality_prior":prior,
                     "hybrid_score":combined, "ranking_mode":"bm25_api_only"})
    return pd.DataFrame(rows).sort_values("hybrid_score", ascending=False).head(top_n).reset_index(drop=True)


## Agent 5 — Passage Extraction & Ranking

Module: `src/agents/agent_05_passage_extraction.py`

In [ ]:
BOILERPLATE_PATTERNS = [
    r"\bstatistical analysis was performed using\b", r"\bdata were analyzed using\b",
    r"\bIBM Corp\b", r"\bSPSS version\b", r"\bthe authors declare no conflict\b",
    r"\bno conflict of interest\b", r"\bwritten informed consent\b",
    r"\bapproved by the institutional review board\b"
]

def split_sentences(text):
    text = clean_text(text)
    if not text:
        return []
    parts = re.split(r"(?<=[.!?])\s+(?=[A-Z0-9])", text)
    out = []
    for sentence in parts:
        sentence = sentence.strip()
        if len(sentence) < 35 or len(sentence) > 900:
            continue
        if any(re.search(pattern, sentence, re.I) for pattern in BOILERPLATE_PATTERNS):
            continue
        out.append(sentence)
    return out

def context_flags(sentence):
    t = sentence.lower()
    flags = []
    if re.search(r"\b(respondents|participants|survey|questionnaire|believed|perceived|awareness|knowledge)\b", t):
        flags.append("survey_or_belief_context")
    if re.search(r"\b(may|might|could|possibly|associated with|in this study|suggests?)\b", t):
        flags.append("uncertainty_or_study_context")
    if re.search(r"\b(secondary bacterial|co-infection|coinfection|superinfection)\b", t):
        flags.append("secondary_infection_context")
    if re.search(r"\b(prescribing|prescribed|prescription|antibiotic use|stewardship)\b", t):
        flags.append("use_or_prescribing_context")
    if re.search(r"\b(in vitro|mouse model|mice|murine|cell line|animal model)\b", t):
        flags.append("preclinical_context")
    if re.search(r"\b(review|systematic review|meta-analysis|survey|case report|editorial|guideline)\b", t):
        flags.append("study_type_may_matter")
    return "|".join(flags)

def _claim_overlap(claim, sentence):
    claim_tokens = {t for t in tokenize(claim) if t not in STOPWORDS and len(t) > 2}
    sent_tokens = set(tokenize(sentence))
    if not claim_tokens:
        return 0.0
    return len(claim_tokens & sent_tokens) / len(claim_tokens)

def rank_candidate_passages(claim, ranked_papers, top_n=MAX_EVIDENCE_PASSAGES):
    if ranked_papers.empty:
        return pd.DataFrame()
    candidates = []
    for _, paper in ranked_papers.iterrows():
        for j, sentence in enumerate(split_sentences(paper.get("abstract", ""))):
            flags = context_flags(sentence)
            candidates.append({
                "claim": claim, "source_id": paper.get("source_id", ""),
                "title": paper.get("title", ""), "year": paper.get("year", ""),
                "journal": paper.get("journal", ""), "publication_types": paper.get("publication_types", ""),
                "pmid": paper.get("pmid", ""), "pmcid": paper.get("pmcid", ""),
                "doi": paper.get("doi", ""), "url": paper.get("url", ""),
                "candidate_sentence": sentence, "sentence_index": j,
                "paper_hybrid_score": float(paper.get("hybrid_score", 0)),
                "context_flags": flags, "claim_token_overlap": _claim_overlap(claim, sentence)
            })
    if not candidates:
        return pd.DataFrame()
    texts = [item["candidate_sentence"] for item in candidates]
    tokenized = [tokenize(item) for item in texts]
    query_tokens = [t for t in tokenize(claim) if t not in STOPWORDS]
    bm = np.zeros(len(texts))
    if any(tokenized) and query_tokens:
        bm = _minmax(np.asarray(BM25Okapi(tokenized).get_scores(query_tokens), dtype=float))
    sem = np.zeros(len(texts))
    sem_used = False
    if embedding_model is not None:
        try:
            qv = embedding_model.encode([claim], normalize_embeddings=True, show_progress_bar=False)[0]
            sv = embedding_model.encode(texts, normalize_embeddings=True, show_progress_bar=False)
            # Keep raw cosine similarity for interpretability; do not min-max stretch weak matches to 1.0.
            sem = np.clip((np.asarray(sv @ qv, dtype=float) + 1.0) / 2.0, 0.0, 1.0)
            sem_used = True
        except Exception as exc:
            print("Sentence embedding fallback:", str(exc)[:180])
    rows = []
    for i, item in enumerate(candidates):
        overlap = item["claim_token_overlap"]
        semantic_score = sem[i] if sem_used else 0.0
        if sem_used:
            score = .30 * bm[i] + .35 * semantic_score + .20 * item["paper_hybrid_score"] + .15 * overlap
        else:
            score = .55 * bm[i] + .25 * item["paper_hybrid_score"] + .20 * overlap
        flags = item["context_flags"]
        if "survey_or_belief_context" in flags: score *= 0.70
        if "preclinical_context" in flags: score *= 0.80
        if "secondary_infection_context" in flags: score *= 0.80
        # Broad generic statements are retained in the audit but rank lower.
        if overlap < 0.15: score *= 0.65
        rows.append({
            **item, "sentence_bm25_norm": float(bm[i]),
            "sentence_semantic_similarity_norm": float(semantic_score),
            "passage_hybrid_score": float(max(0.0, score)), "semantic_used": sem_used
        })
    rows.sort(key=lambda item: item["passage_hybrid_score"], reverse=True)
    chosen, counts = [], {}
    for row in rows:
        sid = row["source_id"]
        # One passage per paper by default prevents one paper dominating the final stance.
        if counts.get(sid, 0) >= 1:
            continue
        chosen.append(row)
        counts[sid] = 1
        if len(chosen) >= top_n:
            break
    return pd.DataFrame(chosen)


## Agent 6 — Gemma 4 Evidence Comparison

Module: `src/agents/agent_06_gemma_nli.py`

In [ ]:
NLI_LOAD_ERROR = ""
def canonical_nli_label(label):
    label = str(label).lower()
    if "entail" in label or "support" in label: return "entailment"
    if "contrad" in label or "refute" in label: return "contradiction"
    if "neutral" in label or "unclear" in label: return "neutral"
    return "unknown"

def _extract_json_object(text):
    text = str(text or "").strip()
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text, flags=re.I)
    match = re.search(r"\{.*\}", text, flags=re.S)
    if not match: raise ValueError("Gemma response did not contain a JSON object.")
    return json.loads(match.group(0))

def check_nli(claim, sentence):
    """Hosted Gemma stance classification. Scores are model estimates, not calibrated truth probabilities."""
    try:
        client = get_genai_client()
        prompt = ("You are an evidence-stance classifier for a scientific literature research prototype.\n"
                  "Compare EVIDENCE PASSAGE (premise) with EXACT CLAIM (hypothesis). Return ONLY a JSON object with keys label (entailment, contradiction, or neutral), confidence (0 to 1, NOT a calibrated probability), and rationale.\n"
                  "Entailment only if the passage directly supports the exact claim; contradiction only if it directly conflicts; neutral if unrelated, indirect, reports prescribing/beliefs rather than efficacy, or lacks context. Preserve negation and distinguish bacterial from viral infections. Do not infer efficacy from prescribing frequency.\n"
                  + "EVIDENCE PASSAGE: " + str(sentence) + "\nEXACT CLAIM: " + str(claim))
        response = client.models.generate_content(model=NLI_MODEL_NAME, contents=prompt)
        obj = _extract_json_object(getattr(response, "text", ""))
        label = canonical_nli_label(obj.get("label", "unknown"))
        if label not in {"entailment", "contradiction", "neutral"}: raise ValueError("Gemma returned an unsupported label.")
        try: confidence = max(0.0, min(1.0, float(obj.get("confidence", 0.0))))
        except Exception: confidence = 0.0
        return {"nli_label":label, "nli_scores":{"model_confidence":confidence}, "nli_confidence":confidence,
                "nli_error":"", "nli_rationale":str(obj.get("rationale", ""))}
    except Exception as exc:
        return {"nli_label":"analysis_error", "nli_scores":{}, "nli_confidence":0.0,
                "nli_error":str(exc)[:500], "nli_rationale":""}

def _antibiotic_viral_direct_stance(claim, passage):
    """Conservative lexical guard for direct antibiotic/viral claims; avoids treating prescribing practice as efficacy."""
    c = re.sub(r"\s+", " ", str(claim).lower())
    p = re.sub(r"\s+", " ", str(passage).lower())
    claim_is_antibiotic_viral = bool(
        re.search(r"\b(antibiotic|antibiotics)\b", c)
        and re.search(r"\b(viral|viruses|virus|influenza|flu|common cold|colds)\b", c)
    )
    if not claim_is_antibiotic_viral:
        return None
    passage_has_antibiotic = bool(re.search(r"\b(antibiotic|antibiotics)\b", p))
    passage_has_target = bool(re.search(r"\b(viral|viruses|virus|influenza|flu|common cold|colds)\b", p))
    # If a passage does not address both the medicine class and the claim's viral target,
    # it cannot be counted as direct evidence for/against this specific claim.
    if not (passage_has_antibiotic and passage_has_target):
        return "neutral"
    # Prescribing frequency is not evidence of antiviral efficacy.
    prescribing_only = bool(re.search(r"\b(prescribed|prescription rate|prescribing|prescriptions)\b", p)
                            and not re.search(r"\b(do not|does not|don't|doesn't|not treat|not work|ineffective|should not|cannot)\b", p))
    if prescribing_only:
        return "neutral"
    # Passage must make an efficacy/treatment statement, not merely mention diagnostic difficulty.
    efficacy_pattern = r"\b(treat|treating|treatment|work against|effective against|recommended to treat|used to treat)\b"
    if not re.search(efficacy_pattern, p):
        return None
    claim_negative = bool(re.search(r"\b(do not|does not|don't|doesn't|cannot|never|not)\b", c))
    passage_negative = bool(re.search(r"\b(do not|does not|don't|doesn't|cannot|never|not|ineffective|should not)\b", p))
    # Positive claim + explicit negative evidence => contradiction; negative claim + explicit negative evidence => entailment.
    if claim_negative and passage_negative:
        return "entailment"
    if not claim_negative and passage_negative:
        return "contradiction"
    if claim_negative and not passage_negative:
        return "contradiction"
    return "entailment"

def nli_evidence_agent(claim, passages):
    if passages.empty: return passages.copy()
    rows=[]
    for _,row in passages.iterrows():
        item=row.to_dict()
        nli = check_nli(claim,item.get("candidate_sentence",""))
        override = _antibiotic_viral_direct_stance(claim,item.get("candidate_sentence",""))
        item.update(nli)
        item["nli_model_label"] = nli.get("nli_label","unknown")
        item["domain_stance_override"] = override or ""
        if override:
            item["nli_label"] = override
            # Use a conservative fixed decision weight for explicit lexical rules; never present it as a probability.
            item["decision_confidence"] = 0.82 if override in {"entailment","contradiction"} else 0.0
            item["stance_method"] = "domain_rule_guard_overrode_nli"
        else:
            item["decision_confidence"] = float(nli.get("nli_confidence",0.0) or 0.0)
            item["stance_method"] = "nli_model"
        rows.append(item)
    return pd.DataFrame(rows)


## Agent 7 — Citation Metadata & Context Audit

Module: `src/agents/agent_07_citation_context_audit.py`

In [ ]:
def citation_context_audit_agent(df):
    if df.empty: return df.copy()
    df=df.copy()
    df["has_source_identifier"]=df.apply(lambda r: bool(str(r.get("pmid","")).strip() or str(r.get("pmcid","")).strip()
        or str(r.get("doi","")).strip() or str(r.get("url","")).startswith("http")),axis=1)
    df["has_passage"]=df["candidate_sentence"].astype(str).str.len().gt(0)
    df["metadata_check"]=np.where(df["has_source_identifier"] & df["has_passage"],
                                  "identifier_and_passage_present_only","missing_identifier_or_passage")
    df["requires_human_context_review"]=df["context_flags"].astype(str).str.len().gt(0)
    df["human_relevance"]=""; df["human_stance"]=""; df["reviewer_rationale"]=""
    # Metadata presence never means the passage has been independently verified.
    df["citation_support_verified"]=False
    # A transparent grouping aid; not a final truth label.
    df["evidence_group"]=df["nli_label"].map({
        "entailment":"provisionally_supports_claim",
        "contradiction":"provisionally_contradicts_claim",
        "neutral":"neutral_or_unclear"
    }).fillna("unclassified")
    df["needs_context_review"]=df["requires_human_context_review"] | df["nli_label"].isin(["analysis_error","model_unavailable"])
    return df


## Agent 8 — Gemma 4 Evidence Synthesis

Module: `src/agents/agent_08_gemma_synthesis.py`

In [ ]:
def extractive_summary(claim, df, max_items=4):
    if df is None or df.empty: return "No candidate passages were retrieved. This does not prove that no evidence exists."
    lines=[]
    for _, r in df.head(max_items).iterrows():
        lines.append(f"- [{r.get('source_id','')}] {r.get('title','')}: {r.get('candidate_sentence','')} (stance signal: {r.get('nli_label','unknown')})")
    return "Extractive overview only; not a factual verdict. Read each passage in context.\\n" + "\\n".join(lines)

def llm_synthesis_agent(claim, df):
    fallback=extractive_summary(claim, df)
    if not LLM_ENABLED:
        return {"summary":fallback,"method":"extractive_fallback_no_api_key","llm_used":False,"llm_status":"disabled_no_api_key","error":"","model":GEMINI_MODEL_NAME,"attempted_models":[]}
    items=[]
    for _, r in df.head(6).iterrows():
        items.append({"source_id":r.get("source_id",""),"title":r.get("title",""),"url":r.get("url",""),"passage":r.get("candidate_sentence",""),"nli_signal":r.get("nli_label","unknown"),"context_flags":r.get("context_flags",""),"stance_method":r.get("stance_method","gemma_api")})
    prompt=("Summarize only supplied evidence passages for a research prototype. Do not add outside facts or provide medical advice. Do not decide whether the claim is true. Distinguish direct evidence from prescribing patterns, survey findings, or indirect context. Mention disagreement and cite source_id for each point. Plain text only.\\nCLAIM: "+claim+"\\nPASSAGES_JSON:\\n"+json.dumps(items,ensure_ascii=False))
    try:
        response=get_genai_client().models.generate_content(model=GEMINI_MODEL_NAME, contents=prompt)
        summary=str(getattr(response,"text","") or "").strip()
        if not summary: raise RuntimeError("Gemma returned an empty synthesis.")
        return {"summary":summary,"method":GEMINI_MODEL_NAME,"llm_used":True,"llm_status":"success","error":"","model":GEMINI_MODEL_NAME,"attempted_models":[{"model":GEMINI_MODEL_NAME,"status":"success"}]}
    except Exception as exc:
        return {"summary":fallback,"method":"extractive_fallback_after_api_error","llm_used":False,"llm_status":"error","error":str(exc)[:700],"model":GEMINI_MODEL_NAME,"attempted_models":[{"model":GEMINI_MODEL_NAME,"status":"error"}]}


## Agent 9 — Conservative Decision & Reporting

Module: `src/agents/agent_09_decision_reporting.py`

In [ ]:
def decision_reporting_agent(claim, scope, retrieval, evidence, synthesis):
    limitations = [
        "Research prototype only; every result requires physician/researcher review.",
        "The direction is provisional and based on retrieved abstract passages plus an NLI model, not proof of truth or clinical efficacy.",
        "NLI scores are model scores, not calibrated probabilities of a claim being true.",
        "Retrieval may miss relevant papers; abstracts may omit important study context.",
        "A citation's existence and metadata do not verify that it supports the sentence shown.",
        "Full-text source, methods, population, limitations, and clinical guidance require human review.",
        "Benchmark labels are reference labels, not independently adjudicated ground truth."
    ]
    empty_lists = {"supporting_papers": [], "contradicting_papers": [], "neutral_or_unclear_papers": [], "context_dependent_papers": []}
    # Claims whose wording explicitly depends on circumstances must not be forced into a binary verdict.
    # This is a wording/logic guard, not a lookup of benchmark IDs or reference labels.
    def _requires_contextual_verdict(text):
        t = re.sub(r"\s+", " ", str(text or "").lower())
        contextual_patterns = [
            r"\bsecondary bacterial\b", r"\bfollowed by\b", r"\bdepends on\b",
            r"\bdepending on\b", r"\bmay require\b", r"\bin some cases\b",
            r"\bunder certain conditions\b", r"\bclinical assessment\b"
        ]
        return any(re.search(p, t) for p in contextual_patterns)
    claim_is_contextual = _requires_contextual_verdict(claim)
    if not scope.get("allowed", False):
        verdict = "Unable to assess this claim because it falls outside the prototype's allowed scope."
        direction, status = "not_assessed", scope.get("status", "blocked")
        reason = scope.get("reason", "Out-of-scope claim.")
        support_papers, oppose_papers, neutral_papers, context_papers = [], [], [], []
    elif retrieval.get("status") == "error":
        verdict = "Unable to assess: literature retrieval failed. This is a technical failure, not evidence for or against the claim."
        direction, status, reason = "technical_failure", "retrieval_error", "Retry retrieval before interpreting the claim."
        support_papers, oppose_papers, neutral_papers, context_papers = [], [], [], []
    elif evidence is None or evidence.empty:
        verdict = "The available search did not return usable evidence for this claim."
        direction, status = "insufficient_evidence", "no_candidate_passages"
        reason = "No usable passages were retained; this does not mean that evidence does not exist."
        support_papers, oppose_papers, neutral_papers, context_papers = [], [], [], []
    else:
        # Aggregate by source first: each paper contributes at most one stance.
        source_rows = {}
        for _, row in evidence.iterrows():
            label = str(row.get("nli_label", "unknown"))
            scores = row.get("nli_scores", {})
            if isinstance(scores, str):
                try:
                    scores = json.loads(scores)
                except Exception:
                    scores = {}
            try:
                confidence = float(row.get("decision_confidence", scores.get(label, row.get("nli_confidence", 0.0))) or 0.0)
            except Exception:
                confidence = 0.0
            relevance = float(row.get("passage_hybrid_score", 0.0) or 0.0)
            flags = str(row.get("context_flags", ""))
            # Discount contextually indirect evidence; do not treat this as source validation.
            context_factor = 1.0
            if "survey_or_belief_context" in flags: context_factor *= 0.45
            if "preclinical_context" in flags: context_factor *= 0.60
            if "secondary_infection_context" in flags: context_factor *= 0.65
            if "uncertainty_or_study_context" in flags: context_factor *= 0.80
            if label not in {"entailment", "contradiction", "neutral"}:
                label = "neutral"
            weight = max(0.0, min(1.0, confidence)) * max(0.0, min(1.0, relevance)) * context_factor
            item = {
                "source_id": str(row.get("source_id", "")), "title": str(row.get("title", "")),
                "year": str(row.get("year", "")), "url": str(row.get("url", "")),
                "pmid": str(row.get("pmid", "")), "pmcid": str(row.get("pmcid", "")),
                "doi": str(row.get("doi", "")), "passage": str(row.get("candidate_sentence", "")),
                "nli_label": label, "nli_confidence": round(confidence, 4),
                "stance_method": str(row.get("stance_method","nli_model")),
                "domain_stance_override": str(row.get("domain_stance_override","")),
                "relevance_score": round(relevance, 4), "context_flags": flags,
                "citation_support_verified": False,
                "human_review_required": True
            }
            sid = item["source_id"] or item["title"]
            # Keep the most relevant/high-confidence passage for each paper.
            rank_key = weight + 0.20 * relevance
            if sid not in source_rows or rank_key > source_rows[sid]["_rank_key"]:
                item["_weight"] = weight
                item["_rank_key"] = rank_key
                source_rows[sid] = item

        support_papers, oppose_papers, neutral_papers, context_papers = [], [], [], []
        support_weight = oppose_weight = neutral_weight = 0.0
        for item in source_rows.values():
            label = item["nli_label"]
            weight = item["_weight"]
            public_item = {k:v for k,v in item.items() if not k.startswith("_")}
            if label == "entailment":
                support_papers.append(public_item); support_weight += weight
            elif label == "contradiction":
                oppose_papers.append(public_item); oppose_weight += weight
            else:
                neutral_papers.append(public_item); neutral_weight += weight
        support_papers.sort(key=lambda x:(x["relevance_score"], x["nli_confidence"]), reverse=True)
        oppose_papers.sort(key=lambda x:(x["relevance_score"], x["nli_confidence"]), reverse=True)
        neutral_papers.sort(key=lambda x:(x["relevance_score"], x["nli_confidence"]), reverse=True)

        active = support_weight + oppose_weight
        total = active + neutral_weight
        ratio = (max(support_weight, oppose_weight) + 1e-9) / (min(support_weight, oppose_weight) + 1e-9)
        # Explicit domain-rule evidence can settle a directly matched claim provisionally;
        # unrelated prescribing/bacterial-treatment passages are neutralized by the guard.
        rule_support = [p for p in support_papers if p.get("stance_method") == "domain_rule_guard_overrode_nli"]
        rule_oppose = [p for p in oppose_papers if p.get("stance_method") == "domain_rule_guard_overrode_nli"]
        if claim_is_contextual:
            # Preserve the conditional nature of the claim, even if retrieved passages lean one way.
            # The interface should explain that applicability depends on clinical context, not assert a universal rule.
            direction = "context_dependent"
            verdict = ("This claim is context-dependent: a viral infection may sometimes be followed by a secondary bacterial infection, "
                       "and antibiotics may be appropriate for the bacterial infection after clinical assessment. "
                       "The retrieved evidence is provisional and does not establish that antibiotics treat the original viral infection.")
            context_papers = support_papers + oppose_papers + neutral_papers
        elif len(source_rows) == 0 or active < 0.08:
            direction = "insufficient_evidence"
            verdict = "The search did not find enough relevant evidence to give a reliable direction."
        elif len(support_papers) >= 2 and support_weight >= 0.10 and support_weight > oppose_weight and ratio >= 1.65:
            direction = "supports_claim"
            verdict = "The retrieved evidence provisionally supports the claim."
        elif len(oppose_papers) >= 2 and oppose_weight >= 0.10 and oppose_weight > support_weight and ratio >= 1.65:
            direction = "contradicts_claim"
            verdict = "The retrieved evidence provisionally contradicts the claim."
        elif support_weight > 0 and oppose_weight > 0 and ratio < 1.65:
            direction = "mixed_or_conflicting"
            verdict = "The retrieved evidence is mixed or conflicting; the claim cannot be resolved reliably from these passages alone."
        elif support_weight > oppose_weight and len(support_papers) >= 1 and support_weight >= 0.13 and (not oppose_papers or ratio >= 1.65):
            direction = "supports_claim"
            verdict = "The retrieved evidence provisionally supports the claim, but independent source confirmation is limited."
        elif oppose_weight > support_weight and len(oppose_papers) >= 1 and oppose_weight >= 0.13 and (not support_papers or ratio >= 1.65):
            direction = "contradicts_claim"
            verdict = "The retrieved evidence provisionally contradicts the claim, but independent source confirmation is limited."
        else:
            direction = "insufficient_evidence"
            verdict = "The retrieved passages are not strong or consistent enough to give a reliable direction."
        status = "awaiting_physician_review"
        reason = (
            f"Paper-level weighted NLI signals (not truth probabilities): supporting={support_weight:.3f}, "
            f"contradicting={oppose_weight:.3f}, neutral/unclear={neutral_weight:.3f}; "
            f"unique sources={len(source_rows)}. Domain-rule overrides are heuristic stance guards, not calibrated probabilities. Check each source in full context."
        )

    counts = evidence["nli_label"].value_counts().to_dict() if evidence is not None and not evidence.empty else {}
    result = {
        "project": "PublicHealth Evidence AI (ABX-AI)", "claim": claim,
        "direct_answer": verdict, "provisional_direction": direction, "verdict": verdict,
        "status": status, "reason": reason, "sources_retrieved": retrieval.get("retrieved_count", 0),
        "candidate_passages": len(evidence) if evidence is not None else 0,
        "nli_signal_counts": counts, "supporting_papers": support_papers,
        "contradicting_papers": oppose_papers, "neutral_or_unclear_papers": neutral_papers,
        "context_dependent_papers": context_papers if "context_papers" in locals() else [],
        "synthesis": synthesis.get("summary", ""), "synthesis_method": synthesis.get("method", ""),
        "llm_used": synthesis.get("llm_used", False), "limitations": limitations,
        "mandatory_review": "A physician/research reviewer must inspect the full cited sources before any conclusion is used.",
        "disclaimer": "Research prototype only. Not for diagnosis, prescribing, or personal treatment decisions."
    }
    return result


## Agent 10 — Supervisor / Orchestrator

Module: `src/agents/agent_10_supervisor_orchestrator.py`

In [ ]:
def supervisor_agent(claim_id, claim, print_steps=True):
    started=datetime.now(timezone.utc).isoformat(); clock=time.time(); trace=[]
    def log(step,details=None):
        trace.append({"step":step,"time_utc":datetime.now(timezone.utc).isoformat(),"details":details or {}})
        if print_steps: print(f"[{len(trace)}] {step}",details or "")
    scope=scope_safety_agent(claim); log("Scope & Safety Agent",scope)
    if not scope["allowed"]:
        retrieval={"status":"not_run","papers":[],"retrieved_count":0,"query_logs":[],"elapsed_seconds":0,"error":""}
        ranked=pd.DataFrame(); passages=pd.DataFrame(); evidence=pd.DataFrame()
        synthesis=llm_synthesis_agent(claim,evidence); result=decision_reporting_agent(claim,scope,retrieval,evidence,synthesis)
        return {"claim_id":claim_id,"claim":claim,"started_at_utc":started,"finished_at_utc":datetime.now(timezone.utc).isoformat(),
          "elapsed_seconds":round(time.time()-clock,3),"scope":scope,"queries":[],"retrieval":retrieval,
          "ranked_papers":ranked,"passages":passages,"evidence_analysis":evidence,"synthesis":synthesis,
          "result":result,"trace":trace,"environment":ENVIRONMENT}
    queries=query_planning_agent(claim); log("Query Planning Agent",{"queries":queries})
    retrieval=research_retriever_agent(queries); log("Europe PMC Retrieval Agent",{"status":retrieval["status"],"retrieved_count":retrieval["retrieved_count"],"query_logs":retrieval["query_logs"]})
    ranked=hybrid_rank_papers(claim,retrieval["papers"]); log("Hybrid Ranking Agent",{"ranked_papers":len(ranked),"mode":ranked["ranking_mode"].iloc[0] if not ranked.empty else "none"})
    passages=rank_candidate_passages(claim,ranked); log("Passage Extraction Agent",{"candidate_passages":len(passages)})
    evidence=citation_context_audit_agent(nli_evidence_agent(claim,passages)); log("NLI + Citation/Context Audit Agents",{"analyzed":len(evidence),"nli_counts":evidence["nli_label"].value_counts().to_dict() if not evidence.empty else {}})
    synthesis=llm_synthesis_agent(claim,evidence); log("LLM Synthesis Agent",{"method":synthesis["method"],"llm_used":synthesis["llm_used"],"llm_status":synthesis.get("llm_status","unknown"),"model":synthesis.get("model",GEMINI_MODEL_NAME),"error":synthesis.get("error","")})
    result=decision_reporting_agent(claim,scope,retrieval,evidence,synthesis); log("Conservative Decision Agent",{"status":result["status"]})
    return {"claim_id":claim_id,"claim":claim,"started_at_utc":started,"finished_at_utc":datetime.now(timezone.utc).isoformat(),
      "elapsed_seconds":round(time.time()-clock,3),"scope":scope,"queries":queries,"retrieval":retrieval,
      "ranked_papers":ranked,"passages":passages,"evidence_analysis":evidence,"synthesis":synthesis,
      "result":result,"trace":trace,"environment":ENVIRONMENT}
print("Supervisor ready.")


## Agent 11 — Human Review Packet

Module: `src/agents/agent_11_review_packet.py`

In [ ]:
REVIEW_COLUMNS=["claim_id","claim","source_id","title","year","doi","pmid","pmcid","url","candidate_sentence",
 "paper_hybrid_score","passage_hybrid_score","nli_label","nli_scores","context_flags","metadata_check",
 "citation_support_verified","human_relevance","human_stance","reviewer_rationale"]
def create_review_packet(output,path):
    df=output.get("evidence_analysis",pd.DataFrame()).copy()
    if df.empty: df=pd.DataFrame(columns=REVIEW_COLUMNS)
    else:
        df["claim_id"]=output.get("claim_id",""); df["claim"]=output.get("claim","")
        for col in REVIEW_COLUMNS:
            if col not in df.columns: df[col]=""
        df=df[REVIEW_COLUMNS]
    df.to_csv(path,index=False); return df


## Agent 12 — Report Export Helpers

Module: `src/agents/agent_12_report_exports.py`

In [ ]:
def json_default(value):
    if isinstance(value,(np.integer,)): return int(value)
    if isinstance(value,(np.floating,)): return float(value)
    if isinstance(value,(np.bool_,)): return bool(value)
    if isinstance(value,(np.ndarray,)): return value.tolist()
    if isinstance(value,(pd.Timestamp,)): return value.isoformat()
    if pd.isna(value) if not isinstance(value,(dict,list,tuple,set)) else False: return None
    return str(value)
def write_json(path,obj):
    with open(path,"w",encoding="utf-8") as f: json.dump(obj,f,indent=2,ensure_ascii=False,default=json_default)

def create_pdf_report(output,path):
    styles=getSampleStyleSheet()
    doc=SimpleDocTemplate(str(path),pagesize=letter,rightMargin=40,leftMargin=40,topMargin=40,bottomMargin=40)
    story=[]; result=output["result"]
    story += [Paragraph("PublicHealth Evidence AI — Preliminary Claim Report",styles["Title"]),Spacer(1,10)]
    for label,value in [("Claim ID",output.get("claim_id","")),("Claim",output.get("claim","")),
                        ("Provisional status",result.get("verdict","")),("Reason",result.get("reason","")),
                        ("Sources retrieved",result.get("sources_retrieved",0)),("Candidate passages",result.get("candidate_passages",0))]:
        story.append(Paragraph(f"<b>{escape(str(label))}:</b> {escape(str(value))}",styles["BodyText"]))
    story += [Spacer(1,10),Paragraph("Candidate evidence (requires human review)",styles["Heading2"])]
    evidence=output.get("evidence_analysis",pd.DataFrame())
    if evidence.empty: story.append(Paragraph("No candidate passages retained. This does not prove that no evidence exists.",styles["BodyText"]))
    else:
        for i,(_,r) in enumerate(evidence.iterrows(),start=1):
            story.append(Paragraph(f"{i}. <b>{escape(str(r.get('title','Untitled')))}</b> ({escape(str(r.get('source_id','')) )})",styles["BodyText"]))
            story.append(Paragraph("NLI signal: "+escape(str(r.get("nli_label","unknown")))+" | Context flags: "+escape(str(r.get("context_flags","") or "none")),styles["BodyText"]))
            story.append(Paragraph("Candidate passage: "+escape(str(r.get("candidate_sentence",""))),styles["BodyText"]))
            if r.get("url"): story.append(Paragraph("Source: "+escape(str(r.get("url",""))),styles["BodyText"]))
            story.append(Spacer(1,7))
    story += [Spacer(1,8),Paragraph("Synthesis",styles["Heading2"]),
              Paragraph(escape(str(result.get("synthesis",""))).replace("\n","<br/>"),styles["BodyText"]),
              Spacer(1,8),Paragraph("Limitations",styles["Heading2"])]
    for item in result.get("limitations",[]): story.append(Paragraph("• "+escape(str(item)),styles["BodyText"]))
    story.append(Paragraph(escape(str(result.get("disclaimer","Research prototype only."))),styles["Italic"]))
    doc.build(story); return path


## Agent 13 — Benchmark Helpers

Module: `src/agents/agent_13_benchmark.py`

In [ ]:
def benchmark_row(output, reference):
    e=output.get("evidence_analysis",pd.DataFrame())
    r=output.get("retrieval",{})
    result=output.get("result",{}) or {}
    synthesis=output.get("synthesis",{}) or {}
    labels=e["nli_label"].value_counts().to_dict() if e is not None and not e.empty and "nli_label" in e.columns else {}
    return {
      "claim_id":reference["claim_id"],"claim":reference["claim"],
      "reference_label":reference["reference_label"],"reference_status":"curated_starter_not_expert_adjudicated",
      "reference_source_url":reference["reference_source_url"],
      "retrieval_status":r.get("status",""),"queries_attempted":len(r.get("query_logs",[])),
      "sources_retrieved":r.get("retrieved_count",0),
      "ranked_papers":len(output.get("ranked_papers",pd.DataFrame())),
      "candidate_passages":len(e) if e is not None else 0,
      "nli_entailment":labels.get("entailment",0),"nli_contradiction":labels.get("contradiction",0),
      "nli_neutral":labels.get("neutral",0),
      "provisional_direction":result.get("provisional_direction",""),
      "direct_answer":result.get("direct_answer",result.get("verdict","")),
      "workflow_status":result.get("status",""),
      "llm_used":bool(synthesis.get("llm_used",False)),
      "llm_status":synthesis.get("llm_status","unknown"),
      "llm_method":synthesis.get("method",""),
      "llm_model":synthesis.get("model",GEMINI_MODEL_NAME),
      "llm_error":synthesis.get("error",""),
      "llm_attempted_models":json.dumps(synthesis.get("attempted_models",[]),ensure_ascii=False),
      "elapsed_seconds":output.get("elapsed_seconds",None),
      "gold_relevant_source_ids":""
    }


## Agent 14 — Evaluation Helpers

Module: `src/agents/agent_14_evaluation.py`

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

def recall_at_k(retrieved_source_ids,gold_relevant_source_ids,k=5):
    gold={str(x).strip() for x in gold_relevant_source_ids if str(x).strip()}
    if not gold:return None
    got=[str(x).strip() for x in list(retrieved_source_ids)[:k]]
    return len(set(got)&gold)/len(gold)

def evaluate_reviewed_stance(df,gold_col="human_stance",predicted_col="nli_label"):
    needed={gold_col,predicted_col}
    if not needed.issubset(df.columns): raise ValueError(f"Missing columns: {needed-set(df.columns)}")
    valid={"supports_claim","contradicts_claim","neutral_or_unclear"}
    clean=df.copy()
    clean[gold_col]=clean[gold_col].astype(str).str.strip().str.lower()
    clean[predicted_col]=clean[predicted_col].astype(str).str.strip().str.lower()
    clean=clean[clean[gold_col].isin(valid)&clean[predicted_col].isin({"entailment","contradiction","neutral"})]
    mapping={"supports_claim":"entailment","contradicts_claim":"contradiction","neutral_or_unclear":"neutral"}
    if clean.empty:return {"status":"not_calculated","reason":"No valid human-reviewed stance labels."}
    y=clean[gold_col].map(mapping); pred=clean[predicted_col]
    return {"status":"calculated_from_reviewed_rows","n":len(clean),
      "accuracy":float(accuracy_score(y,pred)),
      "macro_f1":float(f1_score(y,pred,average="macro",zero_division=0)),
      "report":classification_report(y,pred,zero_division=0,output_dict=True)}

def calculate_benchmark_metrics(results_df):
    """Operational metrics always; reference agreement is exploratory because starter labels are not expert-adjudicated."""
    if results_df is None or results_df.empty:
        return {"operational":{"status":"not_calculated","reason":"No benchmark rows."},
                "claim_reference_agreement":{"status":"not_calculated","reason":"No benchmark rows."},
                "reviewed_gold_metrics":{"status":"not_calculated","reason":"No human-reviewed labels."}}
    n=len(results_df)
    elapsed=pd.to_numeric(results_df.get("elapsed_seconds",pd.Series(dtype=float)),errors="coerce")
    retrieval=results_df.get("retrieval_status",pd.Series([""]*n)).astype(str)
    llm_used=results_df.get("llm_used",pd.Series([False]*n)).fillna(False).astype(bool)
    op={
      "status":"calculated",
      "claims_attempted":int(n),
      "retrieval_success_count":int(retrieval.eq("success").sum()),
      "retrieval_success_rate":float(retrieval.eq("success").mean()),
      "mean_sources_retrieved":float(pd.to_numeric(results_df.get("sources_retrieved",0),errors="coerce").mean()),
      "mean_ranked_papers":float(pd.to_numeric(results_df.get("ranked_papers",0),errors="coerce").mean()),
      "mean_candidate_passages":float(pd.to_numeric(results_df.get("candidate_passages",0),errors="coerce").mean()),
      "mean_elapsed_seconds":float(elapsed.mean()) if elapsed.notna().any() else None,
      "llm_success_count":int(llm_used.sum()),
      "llm_success_rate_among_claims":float(llm_used.mean()),
      "llm_error_count":int(results_df.get("llm_status",pd.Series([""]*n)).astype(str).eq("error").sum()),
      "workflow_error_count":int(results_df.get("workflow_status",pd.Series([""]*n)).astype(str).eq("workflow_error").sum())
    }
    # Preserve the context-dependent class instead of silently excluding it from evaluation.
    ref_map={"SUPPORTED":"supports_claim","CONTRADICTED":"contradicts_claim",
             "CONTEXT_DEPENDENT":"context_dependent"}
    evaluable_labels=["supports_claim","contradicts_claim","context_dependent"]
    eval_df=results_df.copy()
    eval_df["expected_direction"]=eval_df.get("reference_label",pd.Series([""]*n)).astype(str).str.upper().map(ref_map)
    eval_df["predicted_direction"]=eval_df.get("provisional_direction",pd.Series([""]*n)).astype(str)
    eval_df=eval_df[eval_df["expected_direction"].notna() & eval_df["predicted_direction"].isin(evaluable_labels)]
    if len(eval_df):
        y=eval_df["expected_direction"]; pred=eval_df["predicted_direction"]
        ref={
          "status":"exploratory_only_curated_starter_labels_not_expert_adjudicated",
          "evaluation_classes":evaluable_labels,
          "n_scored":int(len(eval_df)),"n_abstained_or_mixed":int(n-len(eval_df)),
          "accuracy_on_scored":float(accuracy_score(y,pred)),
          "macro_f1_on_scored":float(f1_score(y,pred,labels=evaluable_labels,average="macro",zero_division=0)),
          "coverage":float(len(eval_df)/n),
          "reference_class_counts":eval_df["expected_direction"].value_counts().to_dict(),
          "prediction_class_counts":eval_df["predicted_direction"].value_counts().to_dict(),
          "classification_report":classification_report(y,pred,labels=evaluable_labels,zero_division=0,output_dict=True)
        }
    else:
        ref={"status":"not_calculated","reason":"No benchmark rows had both a supported reference class and an evaluable prediction.",
             "evaluation_classes":evaluable_labels,"n_scored":0,"n_abstained_or_mixed":int(n),"coverage":0.0}
    return {"operational":op,"claim_reference_agreement":ref,
            "reviewed_gold_metrics":{"status":"not_calculated","reason":"Requires independently reviewed gold labels and relevant source IDs."}}


## 5. Run one claim

Change `CLAIM_TO_CHECK` and run the supervisor. This call uses the hosted Gemma API once per candidate passage plus a synthesis call, so keep the evidence passage count bounded.

In [ ]:
CLAIM_ID = "ABX-001"
CLAIM_TO_CHECK = "Antibiotics do not treat viral infections."
main_output = supervisor_agent(CLAIM_ID, CLAIM_TO_CHECK, print_steps=True)
print(json.dumps(main_output["result"], indent=2, ensure_ascii=False, default=str))

## 6. Export evidence for human review

In [ ]:
review_packet_df = create_review_packet(main_output, OUT_DIR / "evidence_review_packet.csv")
main_output["ranked_papers"].to_csv(OUT_DIR / "ranked_evidence.csv", index=False)
main_output["evidence_analysis"].to_csv(OUT_DIR / "evidence_analysis.csv", index=False)
print("Outputs saved to:", OUT_DIR.resolve())

## 7. Notes and limitations

- Europe PMC retrieval and BM25 ranking are retained from the source notebook.
- Semantic embedding ranking is disabled to avoid local downloads; retrieval ranking therefore uses BM25 plus a small transparent metadata prior.
- Gemma NLI labels and confidence are model-generated judgments, not calibrated probabilities.
- Every result requires a researcher to inspect the original study, methods, population, and limitations.
- Starter benchmark labels are not expert-adjudicated gold labels.
- API availability, rate limits, and charges depend on your Google AI Studio account.
